In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q unstructured
!pip install -q python-docx
!pip install -q pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-sdk~=1.38.0,

In [4]:
folder_path = "/content/drive/MyDrive/Rag Documents"

In [6]:
!pip install docx2txt


In [7]:
import os
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader

documents = []

folder_path = "/content/drive/MyDrive/Rag Documents"

for file in os.listdir(folder_path):

    file_path = os.path.join(folder_path, file)

    if file.endswith(".pdf"):
        loader = PyPDFLoader(file_path)
        documents.extend(loader.load())

    elif file.endswith(".docx"):
        loader = Docx2txtLoader(file_path)
        documents.extend(loader.load())

print("Total documents loaded:", len(documents))


Total documents loaded: 7


In [8]:
!pip install -q langchain-text-splitters

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

chunks = text_splitter.split_documents(documents)

print("Total chunks created:", len(chunks))

Total chunks created: 7


In [10]:
print(chunks[0].page_content)
print(chunks[0].metadata)

Document 1 — Leave Policy



Employees are entitled to 18 days of paid annual leave per year.
Unused leave can be carried forward to the next year up to a maximum of 10 days.

Employees must apply for leave through the HR portal at least 2 days in advance, except in emergencies.
{'source': '/content/drive/MyDrive/Rag Documents/Document 1 _ Leave Policy.docx'}


In [11]:
!pip install -q sentence-transformers
!pip install -q faiss-cpu

In [12]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_10620/2671871813.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
from langchain_community.vectorstores import FAISS

vector_db = FAISS.from_documents(chunks, embeddings)

In [14]:
vector_db.save_local("hr_vector_db")

In [16]:
retriever = vector_db.as_retriever()

query ="How many leave days are allowed per year?"

#results = retriever.get_relevant_documents(query)
results = retriever.invoke(query)

print(results[0].page_content)

Document 1 — Leave Policy



Employees are entitled to 18 days of paid annual leave per year.
Unused leave can be carried forward to the next year up to a maximum of 10 days.

Employees must apply for leave through the HR portal at least 2 days in advance, except in emergencies.


In [17]:
retriever = vector_db.as_retriever()

query ="What is the sick leave policy?"

#results = retriever.get_relevant_documents(query)
results = retriever.invoke(query)

print(results[0].page_content)

Document 2 — Sick Leave Policy



Employees are entitled to 10 sick leave days per year.

If sick leave exceeds 3 consecutive days, the employee must submit a medical certificate.

Unused sick leave cannot be carried forward to the next year


In [20]:
while True:
    query = input("Ask your question (type 'exit' to stop'): ")

    if query.lower() == "exit":
        break

    retrieved_docs = retriever.invoke(query)
    context = retrieved_docs[0].page_content

    # simple answer extraction
    sentences = context.split(".")

    found = False
    for sentence in sentences:
        if any(word in sentence.lower() for word in query.lower().split()):
            print("Answer:", sentence.strip())
            found = True
            break

    if not found:
        print("Answer:", context.strip())

Ask your question (type 'exit' to stop'): What are working hours?
Answer: Document 6 — Working Hours Policy

The standard working hours are 9:00 AM to 6:00 PM, Monday to Friday
Ask your question (type 'exit' to stop'):  Can I carry forward unused leaves?
Answer: Document 1 — Leave Policy



Employees are entitled to 18 days of paid annual leave per year
Ask your question (type 'exit' to stop'): Is prior approval required for leave?
Answer: Unused leave can be carried forward to the next year up to a maximum of 10 days
Ask your question (type 'exit' to stop'): exit
